## Observability architecture: tracing, spans, structured logging, cost attribution

## Observability is the ability to understand what happened inside your AI system by collecting and analyzing telemetry such as traces, logs, metrics, costs, and execution details.
When something goes wrong, questions arise such as:

Why did the model answer incorrectly?
Which document was retrieved?
Which tool failed?
How much did this request cost?
Why was this request slow?
How many tokens were used?
Which step consumed the most time?

Observability provides visibility into every part of an AI application's execution.
1>Logs
Human-readable records of events.
2>Metrics
Numerical measurements over time.
3>Traces
A trace records the complete execution path of a single request.

=>What is a Trace?
A trace is the container that groups all related operations performed for one request.

Example:
Trace
│
├── Retrieval
├── Prompt
├── GPT
├── Parsing
└── Final Answer

What is a Span?

A span represents a single operation within a trace.
Trace

├── Span: Retrieve Docs
│
├── Span: Prompt Builder
│
├── Span: GPT Call
│
└── Span: Parser

Structured Logging

Instead of free-form text logs, use structured logs.
{
  "timestamp": "2026-07-01T10:15:00Z",
  "level": "ERROR",
  "component": "Retriever",
  "query": "What is LangGraph?",
  "error": "Timeout",
  "duration_ms": 3200,
  "trace_id": "abc123",
  "span_id": "span789"
}
general: Retriever failed

In [28]:
from dotenv import load_dotenv
import os

load_dotenv()

public_key = os.getenv("LANGFUSE_PUBLIC_KEY")
secret_key = os.getenv("LANGFUSE_SECRET_KEY")
host = os.getenv("LANGFUSE_HOST")
print("loaded")

loaded


In [29]:
from langfuse import Langfuse

langfuse = Langfuse(
    public_key=public_key,
    secret_key=secret_key,
    host=host,
)
print(langfuse)

In [3]:
# Create Your Observation
with langfuse.start_as_current_observation(
    name="Application Startup",
    as_type="span",
) as obs:

    print("Langfuse Connected")

    obs.update(
        output="Startup Successful"
    )

Langfuse Connected


In [4]:
from langfuse import Langfuse

langfuse = Langfuse()

print(dir(langfuse))

['__annotations__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_base_url', '_create_experiment_run_name', '_create_observation_from_otel_span', '_create_observation_id', '_create_remote_parent_span', '_create_span_with_parent_context', '_create_trace_tags_via_ingestion', '_environment', '_fetch_prompt_and_update_cache', '_format_otel_span_id', '_format_otel_trace_id', '_get_bounded_max_retries', '_get_current_otel_span', '_get_otel_span_id', '_get_otel_trace_id', '_get_project_id', '_get_span_class', '_is_valid_span_id', '_is_valid_trace_id', '_mask', '_otel_tracer', '_process_experiment_item', '_project_id', '_release', '_resources', '_run_experiment_async', '_start_as_cur

In [5]:
#  a helper function to understand and excute with the retriever
import time
from contextlib import contextmanager

@contextmanager
def observe_step(name: str, metadata: dict = None):
    """
    Creates a Langfuse observation around a block of code.
    Records execution time and metadata.
    """

    start = time.time()

    with langfuse.start_as_current_observation(
        name=name,
        as_type="span",
    ) as obs:

        try:
            yield obs

            duration = round(time.time() - start, 3)

            obs.update(
                metadata={
                    "latency_seconds": duration,
                    **(metadata or {})
                }
            )

        except Exception as e:

            obs.update(
                level="ERROR",
                status_message=str(e)
            )

            raise

In [6]:
from langchain_community.document_loaders import PyPDFLoader

with observe_step("Document Loading") as obs:

    loader = PyPDFLoader("/home/sathw/bootcamp-project/data/Artificial-Intelligence-in-Healthcare.pdf")

    documents = loader.load()

    obs.update(
        output=f"{len(documents)} documents loaded"
    )
print("doc loaded")

/tmp/ipykernel_1658/1824959724.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


doc loaded


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
with observe_step(
    "Chunking",
    metadata={
        "chunk_size":500,
        "chunk_overlap":100
    }
) as obs:

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=100
    )

    chunks = splitter.split_documents(documents)

    obs.update(
        output=f"{len(chunks)} chunks created"

    )
    print(f"length of chunks:{len(chunks)}")

length of chunks:213


In [24]:
import torch

print(torch.__version__)
print(torch.__file__)

2.12.1+cu130
/home/sathw/.pyenv/versions/3.11.9/lib/python3.11/site-packages/torch/__init__.py


In [25]:
# from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

with observe_step("Embedding Model") as obs:

    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    obs.update(
        output="Embedding model initialized"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 406.89it/s]


In [26]:

from langchain_chroma import Chroma
with observe_step("Vector Store Creation") as obs:

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings
    )

    obs.update(
        output="Vector Store Created"
    )

In [27]:
with observe_step("Retriever Initialization") as obs:

    retriever = vector_store.as_retriever(
        search_kwargs={"k": 3}
    )

    obs.update(
        output="Retriever initialized",
        metadata={
            "top_k": 3
        }
    )

In [28]:
query = "What are Advantages of AI in healthcare?"

with observe_step("Document Retrieval") as obs:

    docs = retriever.invoke(query)

    obs.update(
        input=query,
        output=f"Retrieved {len(docs)} documents",
        metadata={
            "retrieved_docs": len(docs)
            }
        )

if len(docs) == 0:
    # langfuse.flush()
    answer = "I couldn't find the answer in the provided documents."
    print(answer)

In [29]:
with observe_step("Context Creation") as obs:

    context = "\n\n".join([doc.page_content for doc in docs])

    obs.update(
        output="Context created",
        metadata={
            "context_length": len(context)
        }
    )

In [9]:
import os
from dotenv import load_dotenv


# AWS Credentials
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_REGION = os.getenv("AWS_REGION")



In [10]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="amazon.nova-micro-v1:0",
    region_name=AWS_REGION,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
)

In [43]:
with observe_step("Prompt Construction") as obs:

    prompt = f"""
You are a helpful AI assistant.

Use ONLY the following context to answer the question.

Context:
{context}

Question:
{query}
"""

    obs.update(
        input=query,
        output="Prompt Created",
        metadata={
            "prompt_length": len(prompt)
        }
    )

In [44]:
with observe_step("LLM Generation") as obs:

    response = llm.invoke(prompt)

    obs.update(
        input=query,
        output=response.content,
        metadata={
            "model": "amazon.nova-micro-v1:0"
        }
    )
    # langfuse.score_current_trace(
    # name="groundedness",
    # value=1
# )

In [46]:
import json

def evaluate_answer(question, context, answer):

    evaluation_prompt = f"""
You are an expert evaluator.

Question:
{question}

Retrieved Context:
{context}

Generated Answer:
{answer}

Evaluate the answer on the following metrics.

Return ONLY valid JSON.

{{
    "groundedness": <0-1>,
    "relevance": <0-1>,
    "correctness": <0-1>,
    "reason": "<short explanation>"
}}
"""

    result = llm.invoke(evaluation_prompt)

    return json.loads(result.content)

In [47]:
langfuse.flush()

In [48]:
print(response.content)

The provided context does not explicitly list the advantages of AI in healthcare. However, it does highlight the critical aspects of ensuring AI systems are accurate, complete, and unbiased to avoid poor decisions and harm. Additionally, it mentions the necessity for compliance with data protection and cybersecurity laws if AI systems use personal data.

From the broader understanding of AI applications in healthcare, common advantages often cited include:

1. **Improved Diagnostic Accuracy**: AI can analyze complex medical data more efficiently and accurately than humans, leading to earlier and more precise diagnoses.
2. **Enhanced Treatment Plans**: AI systems can provide personalized treatment plans by analyzing patient data and predicting outcomes based on historical data.
3. **Efficiency and Cost Reduction**: AI can streamline administrative tasks and reduce the time healthcare providers spend on paperwork, allowing more time for patient care.
4. **Predictive Analytics**: AI can p

In [39]:
langfuse.flush()

In [49]:
def ask_rag(question: str):

    # ---------------------------
    # 1. Retrieve Documents
    # ---------------------------
    with observe_step("Document Retrieval") as obs:

        docs = retriever.invoke(question)

        obs.update(
            input=question,
            output=f"Retrieved {len(docs)} documents",
            metadata={
                "retrieved_docs": len(docs)
            }
        )

    # ---------------------------
    # 2. Create Context
    # ---------------------------
    with observe_step("Context Creation") as obs:

        context = "\n\n".join(
            doc.page_content for doc in docs
        )

        obs.update(
            output="Context Created",
            metadata={
                "context_length": len(context)
            }
        )

    # ---------------------------
    # 3. Prompt Construction
    # ---------------------------
    with observe_step("Prompt Construction") as obs:

        prompt = f"""
You are a helpful AI Assistant.

Answer the question only using the context below.

Context:
{context}

Question:
{question}
"""

        obs.update(
            output="Prompt Created",
            metadata={
                "prompt_length": len(prompt)
            }
        )

    # ---------------------------
    # 4. LLM Generation
    # ---------------------------
    with observe_step("LLM Generation") as obs:

        response = llm.invoke(prompt)

        obs.update(
            input=question,
            output=response.content,
            metadata={
                "model": "amazon.nova-micro-v1:0"
            }
        )
    

# ----------------------------
# Evaluation
# ----------------------------

    with observe_step("Evaluation") as obs:

        evaluation = evaluate_answer(
            query,
            context,
            response.content
        )

        obs.update(
            output=evaluation
        )

        langfuse.create_score(
            trace_id=langfuse.get_current_trace_id(),
            name="groundedness",
            value=evaluation["groundedness"],
            comment=evaluation["reason"]
        )

        langfuse.create_score(
            trace_id=langfuse.get_current_trace_id(),
            name="relevance",
            value=evaluation["relevance"]
        )

        langfuse.create_score(
            trace_id=langfuse.get_current_trace_id(),
            name="correctness",
            value=evaluation["correctness"]
        )

    langfuse.flush()

    return response.content

In [50]:
answer = ask_rag("What is RAG?")
print(answer)

The context provided does not mention "RAG," and there is no information about what RAG refers to within the given text. Therefore, based on the provided context, I cannot determine what RAG is.


## ReAct with langfuse

In [30]:
from langfuse import Langfuse

langfuse = Langfuse()

In [31]:
from langfuse.langchain import CallbackHandler
langfuse_handler = CallbackHandler()

In [32]:
import sqlite3
con = sqlite3.connect("employee.db")
 
cursor=con.cursor()

cursor.execute(""" CREATE TABLE IF NOT EXISTS employee( 
              employee_id INTEGER,
              employee_name TEXT,
              manager TEXT
              )
            """)
cursor.execute("DELETE FROM employee")
cursor.execute(""" INSERT INTO employee VALUES (99, 'vikas' ,'sam Altman')
               """)

cursor.execute(""" INSERT INTO employee VALUES (100, 'mike' ,'Satya Nadella')
               """)

con.commit()
con.close()
print("database ready")

database ready


In [33]:
def db_query(employee_id):
    con = sqlite3.connect("employee.db")
    cursor= con.cursor()
    cursor.execute("""
                SELECT employee_name, manager 
                FROM employee WHERE employee_id=? """, (int(employee_id),)
                )
    result =  cursor.fetchone()

    con.close()

    if result:
        return  (
            f"Employee:{result[0]}, "
            f"Manager:{result[1]}"
        )
    return "Employee not found"

In [34]:
from ddgs import DDGS
def web_search(query):
    with DDGS() as ddgs:
        result =list(ddgs.text(query, max_results=5))

    print(result)
    
    if len(result)==0:
        return "NO result found"
    return result[0]["body"]
    # print(result)

In [35]:
import time
def retry(func):
    def grace(*args):
        retry=3
        
        for attempt in range(retry):
            try:
                result=func(*args)
                return result
            except Exception as e:
                print(f"Retry {attempt +1}")
                time.sleep(1)
        
        return "Tool Failed"
    return grace

In [36]:
db_query_retry = retry(db_query)
web_search_retry=retry(web_search)

In [37]:
from langchain.tools import tool
@tool
def db_query_tool(employee_id: int)-> str:
    """ retrieve Employee information from database"""
    return db_query_retry(employee_id)
@tool
def web_search_tool(query:str)->str:
    """ search in the internet for inforamtion"""
    return web_search_retry(query)


In [38]:
print(web_search_tool.invoke("Sam Altman current company?"))

[{'title': 'Sam Altman - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/Sam_Altman', 'body': 'Sam Altman has recently expanded his investment portfolio to include stakes in over 400 companies, valued at around $2.8 billion.'}, {'title': 'Who is Sam Altman?', 'href': 'https://www.lxahub.com/stories/who-is-sam-altman', 'body': 'In addition to investing and working with tech companies, Altman is an active philanthropist and mentor to young entrepreneurs.'}, {'title': 'Sam Altman’s Startup Portfolio: 14 Companies Backed by the', 'href': 'https://observer.com/2025/06/sam-altman-startup-investments/', 'body': 'Although OpenAI is currently valued at a staggering $300 billion, Altman has stated he holds no equity in the company and receives only a modest ...'}, {'title': 'Sam Altman invested $180 million into a company trying to delay', 'href': 'https://www.technologyreview.com/2023/03/08/1069523/sam-altman-investment-180-million-retro-biosciences-longevity-death/', 'body': 'All these comp

In [39]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[
        db_query_tool,
        web_search_tool
    ],
    system_prompt="""
    You are a ReAct agent.
    Use tools which are required.
    Think step-by-step before answering.
    """
)

In [40]:
from langfuse import observe

@observe(name="ReAct Agent")
def Agent(question):

    response = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        },
        config={
            "callbacks": [langfuse_handler]
        }
    )

    return response["messages"][-1].content

In [41]:
print(Agent("who manages employee 100?"))

<thinking> The tool has returned the manager information for employee 100. The manager is Satya Nadella. I should provide this information to the User.</thinking>

The manager of employee 100 is Satya Nadella. If you need more information about Satya Nadella, please let me know!


In [42]:
langfuse.flush()

1-> why rag is implemented with observe and why not ReAct?
-Because RAG is our own pipeline. Langfuse doesn't know that these are separate stages unless we tell it. The ReAct agent created with LangChain already has an execution graph.